# AMPS I · Week 07 — HW7: Theoretical vs observed convergence rates (a report)
**Due:** before the Week 8 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran. For every rate: **prediction first**, then measurement, then explanation.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Proofs (25 pts)

**(a) (8) Weak derivatives and $H^1$.** (i) Show that $|x|$ has the weak derivative $\operatorname{sign}(x)$ on $(-1,1)$ and that $\operatorname{sign}(x)$ has no
weak derivative in $L^1_{\rm loc}(-1,1)$. (ii) On the quarter disk $D=\{r<1,\ 0<\theta<\pi/2\}$ show that $r^\alpha\in H^1(D)$ if and only if $\alpha>0$
(for $\alpha\le0$ also check $L^2$). (iii) A piecewise linear function that jumps across one mesh edge: is it in $H^1$? Why does this matter for the choice of $V_h$?

**(b) (10) Aubin–Nitsche with Dirichlet conditions.** $-\Delta u=f$ in $\Omega$, $u=0$ on $\partial\Omega$, P$k$ elements on $V_h\subset H^1_0$.
(i) Write the dual problem and prove $\|u-u_h\|_{L^2}\le Ch\,|u-u_h|_{H^1}$, stating exactly where convexity of $\Omega$ is used.
(ii) If $\Omega$ has a re-entrant corner with $\lambda=\pi/\omega<1$, the dual solution is only in $H^{1+\lambda-\varepsilon}$. What do you get instead,
and what are the predicted $L^2$ rates for P1 on the L-shaped domain and on a domain with $\omega=5\pi/4$?

**(c) (7) Zero error (Exercise 5.47).** On $(0,1)$, P1 with $N$ cells: find $u_h\in V_h$ with $\int_0^1vu_h+v'u_h'=\int_0^1vx+v(1)-v(0)$ for all
$v\in V_h$. Show that $\|u-u_h\|_{H^1}=0$ for every $N$ (find $u$ first) and explain with Céa's lemma.

> *your proofs*

## Q2 — Rate report: smooth vs singular solutions (40 pts)
Three problems, all $-\Delta u=f$ with Dirichlet data from the exact solution:

| | domain | exact solution |
|---|---|---|
| S | unit square | $\sin\pi x\sin\pi y$ |
| L | $[-1,1]^2$ minus the quadrant $x>0,y<0$ ($\omega=3\pi/2$) | $r^{2/3}\sin(2\theta/3)$ |
| V | $[-1,1]^2$ minus $\{y<0,\ y<x\}$ ($\omega=5\pi/4$) | $r^{4/5}\sin(4\theta/5)$, $\theta\in[0,5\pi/4]$ from the positive $x$-axis |

**(a) (8) Predict** before you compute: for each problem, the Sobolev regularity of $u$ and the $H^1$ and $L^2$ rates of P1, P2, P3. One table.

**(b) (14) Measure** with `-np 2`, P1–P3, $h=\tfrac14,\dots,\tfrac1{32}$ (S: $N=4,\dots,32$ on the unit square). Report errors, consecutive rates and a
least-squares rate over the last three meshes, and one log-log plot per problem with reference slopes. Use the helper functions provided
(`corner_domain`, `corner_exact`) or your own.

**(c) (10) Graded meshes for V.** Choose $\mu$ for P1 and for P2 from $\mu<\lambda/k$ and refine as in the lab ($H=\tfrac14,\dots,\tfrac1{32}$ for P1,
$\tfrac14,\dots,\tfrac1{16}$ for P2). Report the slopes against $N_{\rm dof}$ and plot error vs $N_{\rm dof}$ together with the uniform meshes.

**(d) (8) Conclusions.** Predicted vs observed in one table, every deviation > 0.05 explained, and three sentences of practical advice.

Helper functions (you may use them unchanged):
```python
def corner_domain(N, omega):
    """[-1,1]^2 (2N x 2N cells, h = 1/N) minus a wedge at the origin: interior angle omega = 5pi/4, 3pi/2 (L-shape) or 7pi/4."""
    big = mesh.create_rectangle(MPI.COMM_WORLD, [[-1, -1], [1, 1]], [2 * N, 2 * N], diagonal=mesh.DiagonalType.right)
    mid = mesh.compute_midpoints(big, 2, np.arange(big.topology.index_map(2).size_local, dtype=np.int32))
    if np.isclose(omega, 1.25 * np.pi):
        out = (mid[:, 1] < 0) & (mid[:, 1] < mid[:, 0])                      # y < 0 and below the line y = x removed
    elif np.isclose(omega, 1.5 * np.pi):
        out = (mid[:, 0] > 0) & (mid[:, 1] < 0)                              # quadrant x > 0, y < 0 removed
    else:
        out = (mid[:, 0] > 0) & (mid[:, 1] > 0) & (mid[:, 1] < mid[:, 0])    # wedge 0 < y < x removed
    return mesh.create_submesh(big, 2, np.where(~out)[0].astype(np.int32))[0]    # (mesh, entity map, vertex map, geom map)

def corner_exact(msh, omega):
    """u = r^lam sin(lam theta), lam = pi/omega, theta measured from one boundary ray of the corner (u = 0 on both rays)."""
    x = ufl.SpatialCoordinate(msh)
    r, th = ufl.sqrt(x[0] ** 2 + x[1] ** 2), ufl.atan2(x[1], x[0])       # atan2 in (-pi, pi]
    if omega < 1.6 * np.pi:
        th = ufl.conditional(ufl.lt(th, 0), th + 2 * ufl.pi, th)         # theta in [0, omega] for 5pi/4 and 3pi/2
    else:
        th = ufl.conditional(ufl.lt(th, np.pi / 4 - 1e-14), th + 2 * ufl.pi, th) - np.pi / 4   # [0, 7pi/4] from the ray y = x
    lam = np.pi / omega
    return r ** lam * ufl.sin(lam * th)

def corner_solve(msh, k, omega):
    """-Δu = 0 with u = exact on the boundary; returns (number of DOFs, L2 error, H1 error, u_h)."""
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    u_ex = corner_exact(msh, omega)
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(g, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))
    ndofs = V.dofmap.index_map.size_global
    uh = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, fem.Constant(msh, 0.0) * v * ufl.dx, bcs=[bc],
                       petsc_options_prefix=f"corner_{k}_{ndofs}_", petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    return (ndofs, *errors(msh, uh - u_ex), uh)
def graded(H, mu, omega):
    """Refine the corner mesh (h = H) until h_K <= 1.5 H r^(1-mu), r = distance of the cell midpoint to the corner."""
    msh = corner_domain(int(round(1 / H)), omega)
    for _ in range(40):
        msh.topology.create_entities(1); msh.topology.create_connectivity(2, 1)
        cells = np.arange(msh.topology.index_map(2).size_local, dtype=np.int32)
        hK = msh.h(2, cells)
        r = np.linalg.norm(mesh.compute_midpoints(msh, 2, cells)[:, :2], axis=1)
        mark = cells[hK > 1.5 * H * np.maximum(r, H ** (1 / mu)) ** (1 - mu)]
        if msh.comm.allreduce(len(mark), op=MPI.SUM) == 0:                # collective: every rank decides together
            break
        edges = mesh.compute_incident_entities(msh.topology, mark, 2, 1)
        msh = mesh.refine(msh, edges)[0]                                   # refine returns (mesh, parent cells, parent facets)
    return msh

def dof_slopes(R):
    """Slopes of the errors against the number of DOFs: -d log E / d log N_dof (columns of R: N_dof, L2, H1)."""
    R = np.asarray(R, dtype=float)
    return [np.round(-np.diff(np.log(R[:, j])) / np.diff(np.log(R[:, 0])), 2) for j in (1, 2)]
```

> *your prediction table (a)*

In [ ]:
%%fenicsx -np 2
# TODO: (b) errors and rates for S, L, V with P1-P3; save them to /tmp/fnx/hw7_rates.npz

In [ ]:
# TODO: (b) one log-log plot per problem (H1 and L2 errors of P1-P3 against h) with reference slopes

In [ ]:
%%fenicsx -np 2
# TODO: (c) graded meshes for V: P1 and P2, slopes against N_dof; save to /tmp/fnx/hw7_graded.npz

In [ ]:
# TODO: (c) error against N_dof, uniform vs graded, P1 and P2

> *your conclusions (d)*

## Q3 — Measurement pitfalls (15 pts)

**(a) (9)** $-u''=\chi_{(0,1/2)}$ on $(0,1)$, $u(0)=u(1)=0$, exact $u=-x^2/2+3x/8$ for $x<\tfrac12$ and $(1-x)/8$ for $x>\tfrac12$. Use odd $N=9,17,33,65,129$
(so $x=\tfrac12$ is the midpoint of a cell), the load `chi * v * ufl.dx(metadata={"quadrature_degree": q})` with
`chi = ufl.conditional(ufl.lt(x, 0.5), 1.0, 0.0)`, and $q=$ default (no metadata), $20$, $22$. P1–P3. Report $L^2$/$H^1$ rates and
$\int_0^1\chi$ as computed with each rule. Predict the rates from the regularity of $u$ (Q1/Exercise A.2), and explain each row.
*(Hint: `basix.make_quadrature(basix.CellType.interval, q)` gives the points.)*

**(b) (6)** For $u=\sin\pi x\sin\pi y$ (problem S), P3, $N=8,\dots,128$: the $L^2$ error computed directly and by the expanded formula
$\int u^2-2\int uu_h+\int u_h^2$. Where do they part, and why at that level?

In [ ]:
%%fenicsx -np 1
# TODO: (a) jump source with three quadrature rules

> *your prediction and explanation of (a)*

In [ ]:
%%fenicsx -np 2
# TODO: (b) direct vs expanded L2 error, P3

> *your explanation of (b)*

## Q4 — LLM claims about convergence rates (20 pts)
Ask an LLM (any; paste the answer unchanged):

```text
I solve -Δu = 0 with Dirichlet data on two domains with P1, P2 and P3 Lagrange elements on uniform meshes:
(L) the L-shaped domain [-1,1]^2 minus the quadrant x>0, y<0, with exact solution r^(2/3) sin(2θ/3);
(V) [-1,1]^2 minus the region {y<0, y<x}.
1. What H1 and L2 convergence rates in h should I observe for each element on each domain, and why?
2. Is P3 worth its cost compared with P1 on these domains?
3. How should I grade the mesh towards the corner to recover the optimal rate, and what rate in terms of the number of DOFs do I then get?
```

**(a) (8)** Paste the answer below.
**(b) (12)** Table: every factual claim (rate, exponent, grading rule, cost statement) → correct / incorrect / imprecise, **with the number from your
Q2 that decides it**. Mark claims you cannot check with your data.

```text
# paste here
```

| # | claim (quote) | verdict | evidence (your number) |
|---|---|---|---|
| 1 | | | |

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …